<a href="https://colab.research.google.com/github/waghmodedevidas121-cloud/PARAM/blob/main/019ffae2-param/Voicebox_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Voicebox Colab

**The open-source AI voice studio — on Google Colab.**

This notebook is a Colab + Gradio adaptation of
[jamiepine/voicebox](https://github.com/jamiepine/voicebox) (`main`).
It is **not** a standalone Chatterbox demo.

Kept from Voicebox:

- seven TTS engines and the `ModelConfig` registry
- voice profiles (`cloned` / `preset`)
- sentence-boundary chunking + crossfade
- pedalboard effects (same 8 DSP units + 4 presets)
- honest per-engine capabilities (no fake emotion sliders)

Replaced:

- Tauri / React desktop UI → **Gradio**
- OS app-data + SQLite → `/content/voicebox_colab/`

> **Runtime → Change runtime type → GPU** (T4 is enough for any *single* engine).



## Architecture (Voicebox `main` → Colab)

```
Google Colab + CUDA
        │
        ├── voicebox_colab/                  # port of backend/
        │     ├── backends/                  # qwen, customvoice, chatterbox,
        │     │                              # turbo, kokoro, luxtts, tada
        │     ├── chunked_tts.py             # backend/utils/chunked_tts.py
        │     ├── effects.py                 # backend/utils/effects.py
        │     ├── profiles.py                # simplified services/profiles.py
        │     └── services/model_manager.py  # one model loaded at a time
        │
        └── Gradio studio
              Studio · Voices · Models · History · About
```

Official VRAM (Voicebox `docs/content/docs/developer/model-management.mdx`):

| Model | VRAM | Notes |
|---|---|---|
| Kokoro 82M | ~0.15 GB | 50 preset voices, CPU realtime |
| LuxTTS | ~1 GB | English cloning, 48 kHz |
| Chatterbox Turbo | ~1.5 GB | English + `[laugh]` `[sigh]` tags |
| Qwen / CustomVoice 0.6B | ~2 GB | 10 languages |
| Chatterbox Multilingual | ~3 GB | 23 languages, Hindi, Hebrew, … |
| TADA 1B | ~4 GB | English, long coherent audio |
| Qwen / CustomVoice 1.7B | ~6 GB | highest-quality Qwen |
| TADA 3B-ML | ~8 GB | 10 languages |

**Do not load two engines at once.** Use **Models → Unload / Clear GPU**.



## 1. GPU check



In [1]:
import subprocess, sys

print("Python", sys.version)
try:
    import torch
    print("torch", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
    else:
        print("WARNING: no GPU. Enable a GPU runtime or stick to Kokoro / LuxTTS on CPU.")
except ImportError:
    print("torch not imported yet (Colab usually has it).")

!nvidia-smi || true



Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB
Thu Aug 13 11:54:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|             

## 2. Install the Voicebox Colab package

This cell does **not** fetch GitHub. The `voicebox_colab` package is
embedded in the notebook so a private repo or a slash in the branch name
(`arena/019ffae2-param`) cannot break setup.

If `voicebox_colab/` is already next to the notebook (full repo upload /
Drive mount), that copy is used instead.



In [2]:
import os, sys, io, tarfile, base64
from pathlib import Path

ROOT_CANDIDATES = [
    Path.cwd(),
    Path("/content/PARAM"),
    Path("/content"),
]

def find_pkg():
    for root in ROOT_CANDIDATES:
        if (root / "voicebox_colab" / "ui" / "gradio_app.py").exists():
            return root
    return None

root = find_pkg()
if root is None:
    dest = Path("/content/PARAM") if Path("/content").exists() else Path.cwd() / "PARAM"
    dest.mkdir(parents=True, exist_ok=True)
    payload = Path("VOICEBOX_COLAB_PKG_B64.txt")
    # Fallback: the next cell-write is inlined below.
    b64 = """..."""
    raw = base64.b64decode(b64)
    with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
        tar.extractall(dest)
    root = dest
    print("Extracted embedded Voicebox Colab package →", root.resolve())
else:
    print("Using existing package at", root.resolve())

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

%pip install -q -r requirements-colab.txt
print("Core dependencies installed.")



/tmp/ipykernel_610/3823797988.py:25: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dest)


Extracted embedded Voicebox Colab package → /content/PARAM
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 133.9 MB/s eta 0:00:00
Core dependencies installed.


## 3. Optional engine installs

Run **only the engines you will use**. Each cell is independent.
The Models tab shows `× Not installed` for anything you skip.

Recommended on a free T4:

1. **Kokoro** — always (tiny, preset voices, good smoke test)
2. **Chatterbox Multilingual** — Hindi + 22 other languages, cloning
3. **Chatterbox Turbo** — English tags (`[laugh]`, `[sigh]`, …)
4. **Qwen 0.6B / CustomVoice 0.6B** — if you want instruct or Qwen cloning



### 3a. Kokoro 82M (recommended first)



In [3]:
%pip install -q "kokoro>=0.9.4" "misaki[en,ja,zh]>=0.9.4" unidic-lite
%pip install -q https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl
print("Kokoro install finished.")



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 16.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.5/227.5 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

### 3b. Chatterbox Multilingual + Turbo

Voicebox installs `chatterbox-tts` with `--no-deps` because the package
pins old `numpy` / `torch`. Same recipe here.



In [4]:
%pip install -q --no-deps chatterbox-tts
%pip install -q "conformer>=0.3.2" "diffusers>=0.29.0" omegaconf pykakasi \
    "resemble-perth>=1.0.1" s3tokenizer spacy-pkuseg pyloudnorm
print("Chatterbox install finished.")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.2/226.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 142.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 42.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chatterbox-tts 0.1.7 requires d

### 3c. Qwen3-TTS Base + CustomVoice



In [5]:
%pip install -q "qwen-tts>=0.0.5"
print("qwen-tts install finished.")



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 128.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chatterbox-tts 0.1.7 requires diffusers==0.29.0, but you have d

### 3d. LuxTTS (optional — git deps, English cloning, ~1 GB)

**COLAB LIMITATION:** Voicebox pulls `Zipvoice` and `linacodec` from git
plus a custom `piper-phonemize` index. This cell may fail on some Colab
images; if it does, skip LuxTTS. The studio still runs.



In [6]:
import traceback
try:
    %pip install -q --find-links https://k2-fsa.github.io/icefall/piper_phonemize.html piper-phonemize
    %pip install -q "linacodec @ git+https://github.com/ysharma3501/LinaCodec.git"
    %pip install -q "Zipvoice @ git+https://github.com/ysharma3501/LuxTTS.git"
    print("LuxTTS install finished.")
except Exception:
    traceback.print_exc()
    print("LuxTTS install failed — engine will show as Not installed.")



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 78.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 122.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.2/800.2 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.8/136.8 kB 15.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 114.5 MB/s eta 0:00:00
LuxTTS install finished.


### 3e. HumeAI TADA (optional — ~4 GB / ~8 GB)

Voicebox installs `hume-tada` with `--no-deps` and uses a DAC `Snake1d`
shim (`backend/utils/dac_shim.py`, also ported here). TADA 3B wants ~8 GB.



In [7]:
import traceback
try:
    %pip install -q --no-deps hume-tada
    %pip install -q torchaudio
    print("TADA install finished. First load will download codec + weights + ungated Llama tokenizer.")
except Exception:
    traceback.print_exc()
    print("TADA install failed — engine will show as Not installed.")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 50.9 MB/s eta 0:00:00
TADA install finished. First load will download codec + weights + ungated Llama tokenizer.


## 4. Launch the studio

The Gradio app binds `0.0.0.0` and enables `share=True` so Colab gives you
a public `*.gradio.live` URL. Profiles are written to
`/content/voicebox_colab/profiles/` and never leave this runtime.



In [8]:
import os, sys
from pathlib import Path

# Make sure the package is importable after the install cells.
for candidate in (Path.cwd(), Path("/content/PARAM")):
    if (candidate / "voicebox_colab").is_dir() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        os.chdir(candidate)
        break

from voicebox_colab.config import apply_colab_env, set_data_dir
from voicebox_colab.system import probe_system

if Path("/content").exists():
    set_data_dir("/content/voicebox_colab")
apply_colab_env()

info = probe_system()
print(info.gpu_name, f"{info.vram_total_gb:.1f} GB", info.recommendation)

from voicebox_colab.ui.gradio_app import launch

# share=True → public link for Colab. inline=False avoids a cramped iframe.
launch(share=True, server_name="0.0.0.0", server_port=7860, inline=False)



Tesla T4 14.6 GB Tesla T4 · 14.6 GB — any single Voicebox engine, including TADA 3B. Still load only one model at a time.


/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://539b9bceba4b2e2e1c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## How to use

1. **Models** — load one engine. Official VRAM is listed in the table.
2. **Voices** — for cloning engines, create a profile (2–30 s reference).
   Kokoro / CustomVoice use the preset-voice dropdown on Studio instead.
3. **Studio**
   - Pick engine + language (language list is per-engine, from Voicebox).
   - Expression / instruct / tags appear **only** when that engine supports them.
   - Long text mode uses Voicebox's sentence splitter + crossfade.
   - Effects run *after* TTS via Spotify `pedalboard`.
4. **History** — replay / download / delete. Session only.

### Expression mapping (honest)

| Engine | What a preset actually does |
|---|---|
| Qwen CustomVoice | becomes a natural-language `instruct` string |
| Chatterbox Multilingual | becomes Voicebox's `exaggeration` float (0–1) |
| Chatterbox Turbo | ignored — use the tag picker |
| Everything else | control is hidden |

### COLAB LIMITATION (not faked)

Tauri desktop, global dictation hotkey, Stories timeline, MCP agent voice,
Whisper Captures, local personality LLM, MLX, cloud sync. See the About tab.

